# Check 8: What do the noise labels z control, and what does ε control? (generator analysis)

**Hypothesis (from the architecture and Check 5 R3; not yet tested).**
- ε is a **local** input: it is cut into the same 16×16 patches as the generator's tokens.
- The noise labels z (32 values, each 0–63) are a **global** code: they're added into the 768-d conditioning vector `cond = E_class[c] + Σᵢ Eᵢ[zᵢ] + 0.02·cfg term`, which acts on every token identically (16 concatenated conditioning tokens + AdaLN scale/shift/gate in all 12 blocks and the final layer).
- So z plausibly selects an image-wide **mode** within the class (colour palette, texture family, style), and ε fills in the **spatial arrangement and detail**.

**Two testable claims and three experiments.**

| | Claim tested | How | Prediction if the hypothesis holds |
|---|---|---|---|
| **A** | what each input controls | factorial grid: 8 codes z × 8 ε per class, 8 classes (512 samples); two-way variance decomposition of image descriptors | z explains most variation of **position-independent** descriptors (colour statistics, spatially pooled AlexNet features, power spectrum); ε explains most variation of **position-specific** ones (16×16 thumbnail, edge layout) |
| **B** | locality | perturb ε in **one** 16×16 patch, or **one** coordinate zᵢ, and map where the output changes | ε-patch changes concentrate in that tile and its neighbours; zᵢ changes spread over the whole image |
| **C** | R3 errors are global | per-tile error maps of the cached Check 5 R3 reconstructions (known vs random z) | with random z the error rises in most tiles, not in a few |

**What exactly is done.**
- **No optimization, no HR, no test images.** A and B are forward passes of the frozen generator (CFG 1.0, eager path); C reloads cached Check 5 outputs. Only class *labels* are used.
- **Classes:** the 4 development images' classes (streetcar, bonnet, crane, spoonbill) + 4 more drawn with seed 2034.
- **Seeds:** codes z with seed 2035, ε with 2036 (A); perturbation positions/coordinates with 2037 (B); all folded with the class, so everything is reproducible.
- **Descriptors (A).**
  - Position-independent: per-channel colour mean and std (6-d); spatially pooled, channel-normalized AlexNet features from the LPIPS network (5 layers, 1,152-d; a texture/style summary); log radial power spectrum (32 bins).
  - Position-specific: 16×16 area-downsampled image (768-d, the layout); 32×32 downsampled gradient magnitude (edge layout).
  - Each descriptor dimension is standardized over the 64 samples of a class; then a two-way ANOVA without replication splits the total sum of squares into **z**, **ε** and **interaction** shares (averaged over dimensions, then over classes).
- **Locality (B):** per class, from one base sample (z₀, ε₀): 12 single-patch ε resamples and 12 single-coordinate zᵢ changes. For each, the change energy |Δx|² is summed per 16×16 tile. Metrics: share of change energy in the perturbed tile and in its 3×3 neighbourhood (uniform spread would give 1/256 and 9/256), normalized entropy of the tile-energy map (1 = perfectly uniform), and the mean change magnitude.
- **R3 maps (C):** per-tile MSE between each R3 reconstruction and its generated target x*, for known vs random z (best restart by final LR loss, as in Check 5). Metrics: share of tiles where random-z error exceeds known-z error, and the normalized entropy of the error increase.

**Reading the result.** Both claims can partly fail: attention lets ε patches influence each other, so ε may be partly global. The experiments measure *how* local or global each input is, and any outcome is informative.

**Runtime.** About 10–15 minutes on the GPU (forward passes only; no `platform` allocator needed) plus CPU analysis; cached in `~/work/datasets/ImageNetV2/noise_label_role_dev_v1/`, so re-running only reloads.

## Setup

In [ ]:
import os
import sys
from pathlib import Path

SCRIPTS = Path.home() / "work" / "IADrift" / "scripts"
if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

import io
import json

import lpips
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Markdown, display

# Project helpers (importing run_sr4_csgm_test loads the generator).
from freq_metrics import RADIUS, WINDOW
from run_sr4_csgm_test import DATA_DIR, atomic_write_bytes, generate, jax, jnp, load_index, load_split, model, params
from run_sr4_iadrift_dev import check_dev
from run_sr4_iadrift_smoke import class_name

OUT_DIR = DATA_DIR / "noise_label_role_dev_v1"
OUT_DIR.mkdir(exist_ok=True)
DEV_IMAGES = [8117, 3939, 403, 347]
index_rows = load_index()
labels = np.array([int(row["label"]) for row in index_rows])
train_indices, test_indices = load_split(labels)
check_dev(DEV_IMAGES, train_indices, test_indices)

rng = np.random.default_rng(2034)
dev_classes = [int(labels[i]) for i in DEV_IMAGES]
extra = [int(c) for c in rng.choice(np.setdiff1d(np.arange(1000), dev_classes), size=4, replace=False)]
CLASSES = dev_classes + extra
print("Classes:", [(c, class_name(c)) for c in CLASSES])

torch.set_num_threads(4)
LPIPS_NET = lpips.LPIPS(net="alex", verbose=False).eval()
%matplotlib inline


def sample_eps(seed, *fold):
    key = jax.random.PRNGKey(seed)
    for f in fold:
        key = jax.random.fold_in(key, f)
    return jax.random.normal(key, (1, 256, 256, 3), dtype=jnp.float32)


def sample_z(seed, *fold):
    key = jax.random.PRNGKey(seed)
    for f in fold:
        key = jax.random.fold_in(key, f)
    return jax.random.randint(key, (1, model.noise_coords), 0, model.noise_classes, dtype=jnp.int32)


def gen(eps, label, z):
    # Frozen generator, CFG 1.0, eager path (same as every saved CSGM output).
    return np.asarray(jax.device_get(generate(eps, label, z)), dtype=np.float32)[0]

## A. Factorial z × ε grid and variance decomposition

8 codes × 8 ε per class (64 samples, cached as float16). Descriptors and the two-way decomposition as described above.

In [ ]:
NZ, NE = 8, 8


def grid_for(c):
    path = OUT_DIR / f"grid_class{c:03d}.npz"
    if path.exists():
        with np.load(path) as data:
            return data["x"].astype(np.float32)
    x = np.zeros((NZ, NE, 256, 256, 3), np.float32)
    for k in range(NZ):
        z = sample_z(2035, c, k)
        for j in range(NE):
            x[k, j] = gen(sample_eps(2036, c, j), c, z)
    buffer = io.BytesIO()
    np.savez_compressed(buffer, x=x.astype(np.float16))
    atomic_write_bytes(path, buffer.getvalue())
    return x


grids = {c: grid_for(c) for c in CLASSES}
print("grids:", {c: g.shape for c, g in grids.items()})


def _alex_pooled(x):
    t = torch.from_numpy(np.ascontiguousarray(np.clip(x, -1, 1).transpose(2, 0, 1)[None])).float()
    with torch.no_grad():
        feats = LPIPS_NET.net.forward(LPIPS_NET.scaling_layer(t))
    return np.concatenate([lpips.normalize_tensor(f).mean(dim=(2, 3))[0].numpy() for f in feats])


def _radial_log_power(x, num_bins=32):
    xm = np.asarray(x, np.float64) - np.mean(x, axis=(0, 1))
    p = np.sum(np.abs(np.fft.fft2(xm * WINDOW, axes=(0, 1))) ** 2, axis=-1)
    edges = np.linspace(0.0, 0.5, num_bins + 1)
    return np.log10(np.array([p[(RADIUS >= lo) & (RADIUS < hi)].mean() for lo, hi in zip(edges[:-1], edges[1:])]) + 1e-12)


def _thumb(x, s):
    return x.reshape(s, 256 // s, s, 256 // s, -1).mean(axis=(1, 3))


def _edge_layout(x):
    g = np.abs(np.diff(x.mean(-1), axis=0))[:, :-1] + np.abs(np.diff(x.mean(-1), axis=1))[:-1, :]  # 255x255
    return _thumb(np.pad(g, ((0, 1), (0, 1)))[..., None], 32).ravel()


DESCRIPTORS = {  # name: (kind, function image -> 1-D vector)
    "colour mean/std": ("position-independent", lambda x: np.concatenate([x.mean(axis=(0, 1)), x.std(axis=(0, 1))])),
    "pooled AlexNet features": ("position-independent", _alex_pooled),
    "radial power spectrum": ("position-independent", _radial_log_power),
    "16×16 thumbnail (layout)": ("position-specific", lambda x: _thumb(x, 16).ravel()),
    "edge layout (32×32 gradient)": ("position-specific", lambda x: _edge_layout(x)),
}


def anova_shares(D):
    # D: [NZ, NE, d]. Two-way ANOVA without replication per dimension (dimensions standardized over the 64 samples),
    # summed over dimensions. Returns shares of the total sum of squares: z, eps, interaction.
    D = np.asarray(D, np.float64)
    sd = D.reshape(-1, D.shape[-1]).std(axis=0)
    D = (D - D.reshape(-1, D.shape[-1]).mean(axis=0)) / np.where(sd > 0, sd, 1.0)
    m = D.mean(axis=(0, 1))
    rows, cols = D.mean(axis=1), D.mean(axis=0)
    ss_z = D.shape[1] * np.sum((rows - m) ** 2)
    ss_e = D.shape[0] * np.sum((cols - m) ** 2)
    ss_i = np.sum((D - rows[:, None] - cols[None] + m) ** 2)
    total = ss_z + ss_e + ss_i
    return {"z": ss_z / total, "ε": ss_e / total, "interaction": ss_i / total}


shares = {}
for c in CLASSES:
    for name, (kind, fn) in DESCRIPTORS.items():
        D = np.stack([np.stack([fn(grids[c][k, j]) for j in range(NE)]) for k in range(NZ)])
        shares[(c, name)] = anova_shares(D)

lines = ["| Descriptor | Kind | z share | ε share | interaction | z share per class |", "|---|---|---:|---:|---:|---|"]
for name, (kind, _) in DESCRIPTORS.items():
    s = {k: np.mean([shares[(c, name)][k] for c in CLASSES]) for k in ("z", "ε", "interaction")}
    lines.append(f"| {name} | {kind} | {s['z']:.2f} | {s['ε']:.2f} | {s['interaction']:.2f} | "
                 + ", ".join(f"{shares[(c, name)]['z']:.2f}" for c in CLASSES) + " |")
display(Markdown(f"**Variance decomposition, mean over {len(CLASSES)} classes** (shares of total variation)\n\n" + "\n".join(lines)))

fig, ax = plt.subplots(figsize=(10, 4))
names = list(DESCRIPTORS)
bottom = np.zeros(len(names))
for part, color in (("z", "C0"), ("ε", "C1"), ("interaction", "C7")):
    v = np.array([np.mean([shares[(c, n)][part] for c in CLASSES]) for n in names])
    ax.bar(range(len(names)), v, bottom=bottom, color=color, label=part)
    bottom += v
ax.set_xticks(range(len(names)), [n.replace(" (", "\n(") for n in names], fontsize=8)
ax.set(ylabel="share of variation", title="What varies with the noise labels z vs the noise ε (mean over classes)")
ax.legend()
plt.show()

for c in CLASSES[:3]:
    fig, axes = plt.subplots(NZ, NE, figsize=(NE * 1.9, NZ * 1.9))
    for k in range(NZ):
        for j in range(NE):
            axes[k, j].imshow((np.clip(grids[c][k, j], -1, 1) + 1) / 2)
            axes[k, j].set_xticks([])
            axes[k, j].set_yticks([])
        axes[k, 0].set_ylabel(f"z{k}", fontsize=8)
    for j in range(NE):
        axes[0, j].set_title(f"ε{j}", fontsize=8)
    fig.suptitle(f"Class {c} ({class_name(c)}): rows = noise-label codes z, columns = ε", fontsize=10)
    fig.subplots_adjust(left=0.03, right=0.99, bottom=0.01, top=0.95, wspace=0.03, hspace=0.03)
    plt.show()

## B. Locality: one ε patch vs one noise-label coordinate

Per class, from the base sample (z₀, ε₀) of grid A: 12 single-patch ε resamples and 12 single-coordinate zᵢ changes (seed 2037). Change energy per 16×16 tile.

In [ ]:
NPERT = 12


def tile_energy(dx):
    e = (dx ** 2).sum(axis=-1)
    return e.reshape(16, 16, 16, 16).sum(axis=(1, 3))  # [tile row, tile col]


def norm_entropy(t):
    p = t.ravel() / t.sum()
    p = p[p > 0]
    return float(-(p * np.log(p)).sum() / np.log(t.size))


def locality_for(c):
    path = OUT_DIR / f"locality_class{c:03d}.json"
    if path.exists():
        return json.loads(path.read_text())
    eps0, z0 = sample_eps(2036, c, 0), sample_z(2035, c, 0)
    x0 = gen(eps0, c, z0)
    rng = np.random.default_rng([2037, c])
    out = {"eps_patch": [], "z_coord": []}
    for _ in range(NPERT):
        tr, tc = (int(v) for v in rng.integers(0, 16, size=2))
        new = np.asarray(eps0).copy()
        new[0, 16 * tr:16 * tr + 16, 16 * tc:16 * tc + 16] = rng.standard_normal((16, 16, 3))
        t = tile_energy(gen(jnp.asarray(new), c, z0) - x0)
        nb = t[max(tr - 1, 0):tr + 2, max(tc - 1, 0):tc + 2].sum()
        out["eps_patch"].append({"tile": [tr, tc], "in_tile": float(t[tr, tc] / t.sum()), "in_3x3": float(nb / t.sum()),
                                 "entropy": norm_entropy(t), "rms_change": float(np.sqrt(t.sum() / (256 * 256 * 3))),
                                 "map": t.tolist()})
    for _ in range(NPERT):
        i = int(rng.integers(0, model.noise_coords))
        z = np.asarray(z0).copy()
        z[0, i] = (z[0, i] + int(rng.integers(1, model.noise_classes))) % model.noise_classes  # a different value
        t = tile_energy(gen(eps0, c, jnp.asarray(z)) - x0)
        out["z_coord"].append({"coord": i, "max_tile": float(t.max() / t.sum()), "entropy": norm_entropy(t),
                               "rms_change": float(np.sqrt(t.sum() / (256 * 256 * 3))), "map": t.tolist()})
    path.write_text(json.dumps(out))
    return out


loc = {c: locality_for(c) for c in CLASSES}
ep = [r for c in CLASSES for r in loc[c]["eps_patch"]]
zc = [r for c in CLASSES for r in loc[c]["z_coord"]]
lines = ["| Perturbation | n | share in perturbed tile | share in 3×3 neighbourhood | largest single-tile share | "
         "normalized entropy (1 = uniform) | RMS change |", "|---|---:|---:|---:|---:|---:|---:|",
         f"| one ε patch | {len(ep)} | {np.mean([r['in_tile'] for r in ep]):.3f} | {np.mean([r['in_3x3'] for r in ep]):.3f} | "
         f"{np.mean([max(map(max, r['map'])) / np.sum(r['map']) for r in ep]):.3f} | {np.mean([r['entropy'] for r in ep]):.3f} | "
         f"{np.mean([r['rms_change'] for r in ep]):.4f} |",
         f"| one noise-label coordinate | {len(zc)} | — | — | {np.mean([r['max_tile'] for r in zc]):.3f} | "
         f"{np.mean([r['entropy'] for r in zc]):.3f} | {np.mean([r['rms_change'] for r in zc]):.4f} |",
         f"| uniform spread (reference) | — | {1 / 256:.3f} | {9 / 256:.3f} | {1 / 256:.3f} | 1.000 | — |"]
display(Markdown("**Locality of changes, mean over all perturbations and classes**\n\n" + "\n".join(lines)))

c = CLASSES[0]
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for k in range(4):
    r = loc[c]["eps_patch"][k]
    axes[0, k].imshow(np.log10(np.array(r["map"]) + 1e-9), cmap="magma")
    axes[0, k].add_patch(plt.Rectangle((r["tile"][1] - 0.5, r["tile"][0] - 0.5), 1, 1, fill=False, color="cyan", linewidth=1.5))
    axes[0, k].set_title(f"ε patch {tuple(r['tile'])}: {r['in_3x3']:.0%} in 3×3", fontsize=9)
    r = loc[c]["z_coord"][k]
    axes[1, k].imshow(np.log10(np.array(r["map"]) + 1e-9), cmap="magma")
    axes[1, k].set_title(f"z coordinate {r['coord']}: entropy {r['entropy']:.2f}", fontsize=9)
for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle(f"Class {c} ({class_name(c)}): log change energy per 16×16 tile. Top: one ε patch changed (cyan box). "
             "Bottom: one noise-label coordinate changed.", fontsize=10)
plt.show()

## C. Check 5 R3 reanalysis: are the random-z errors global?

Reloads the cached R3 runs (`representation_error_dev_v1/`) and regenerates the targets x* exactly as in notebook 04 (seed 2032 folded with the image index).

In [ ]:
REP_DIR = DATA_DIR / "representation_error_dev_v1"
lines = ["| Target (class) | mean tile MSE, known z | mean tile MSE, random z | tiles where random z is worse | "
         "normalized entropy of the error increase |", "|---|---:|---:|---:|---:|"]
fig, axes = plt.subplots(3, len(DEV_IMAGES), figsize=(4 * len(DEV_IMAGES), 12))
for col, i in enumerate(DEV_IMAGES):
    label = int(labels[i])
    k_eps, k_nl = jax.random.split(jax.random.fold_in(jax.random.PRNGKey(2032), i))
    eps_star = jax.random.normal(k_eps, (1, 256, 256, 3), dtype=jnp.float32)
    nl_star = jax.random.randint(k_nl, (1, model.noise_coords), 0, model.noise_classes, dtype=jnp.int32)
    x_star = gen(eps_star, label, nl_star)
    tile_mse = {}
    for cond in ("noise_labels_known", "noise_labels_random"):
        runs = []
        for r in (0, 1):
            with np.load(REP_DIR / f"R3_img{i:05d}_{cond}_r{r}.npz") as data:
                runs.append((json.loads(str(data["metrics_json"]))["final_loss"], r, data["x_hat"]))
        x = min(runs, key=lambda t: (t[0], t[1]))[2]
        tile_mse[cond] = tile_energy(x - x_star) / (16 * 16 * 3)
    known, rand = tile_mse["noise_labels_known"], tile_mse["noise_labels_random"]
    inc = np.clip(rand - known, 0, None)
    lines.append(f"| {i} ({class_name(label)}) | {known.mean():.4f} | {rand.mean():.4f} | "
                 f"{np.mean(rand > known):.0%} | {norm_entropy(inc) if inc.sum() > 0 else float('nan'):.2f} |")
    vmax = max(known.max(), rand.max())
    for row, (m, title) in enumerate(((known, "tile MSE, known z"), (rand, "tile MSE, random z"),
                                      (rand - known, "random − known"))):
        lim = np.abs(m).max() if row == 2 else vmax
        axes[row, col].imshow(m, cmap="coolwarm" if row == 2 else "magma", vmin=-lim if row == 2 else 0, vmax=lim)
        axes[row, col].set_title(f"target {i}: {title}", fontsize=9)
        axes[row, col].set_xticks([])
        axes[row, col].set_yticks([])
plt.show()
display(Markdown("**Per-tile errors of in-range SR (Check 5 R3), known vs random noise labels**\n\n" + "\n".join(lines)))

## Save summary

In [ ]:
summary = {
    "classes": CLASSES,
    "anova_shares": {f"{c}|{n}": shares[(c, n)] for c in CLASSES for n in DESCRIPTORS},
    "locality_means": {"eps_patch_in_tile": float(np.mean([r["in_tile"] for r in ep])),
                       "eps_patch_in_3x3": float(np.mean([r["in_3x3"] for r in ep])),
                       "eps_patch_entropy": float(np.mean([r["entropy"] for r in ep])),
                       "z_coord_entropy": float(np.mean([r["entropy"] for r in zc]))},
}
atomic_write_bytes(OUT_DIR / "summary.json", json.dumps(summary, indent=1).encode())
print("Wrote", OUT_DIR / "summary.json")

## Conclusions (8 classes; generator analysis, no HR, no test images)

**The hypothesis is half right: z is global, but it is much more than a "style" code. z is the dominant latent; it selects the whole scene, including layout. ε is a weak, mostly local detail input.**

**A. What each input controls** (share of variation over the 8 × 8 grids, mean over 8 classes):

| Descriptor | z | ε | interaction |
|---|---:|---:|---:|
| colour mean/std (position-independent) | **0.82** | 0.03 | 0.15 |
| radial power spectrum (position-independent) | **0.68** | 0.05 | 0.26 |
| pooled AlexNet features (position-independent) | **0.56** | 0.06 | 0.38 |
| 16×16 thumbnail (layout) | **0.44** | 0.10 | 0.46 |
| edge layout | **0.37** | 0.10 | 0.53 |

- As predicted, z dominates the position-independent descriptors (palette, spectrum, texture/style), in every class.
- **Not as predicted:** z also explains 4× more of the *layout* variation than ε does (0.44 vs 0.10). The rest is interaction: what ε does depends on z.
- The grids show it directly (e.g. crane): each **row** (one z) is a different scene type: a flock in grass, a single bird by water, a close-up head, white cranes in a park, a crowned crane. Each **column** (one ε) only shifts pose and positions within that scene.

**B. Locality** (96 perturbations each):
- Changing **one ε patch**: 5.9% of the change lands in that tile and 19% in its 3×3 neighbourhood (15× and 5× uniform spread). So ε acts locally, but most of the change (81%) still spreads through attention (entropy 0.71). And the change is small: RMS 0.046.
- Changing **one of the 32 noise-label coordinates**: the change is spread over the whole image (entropy 0.90; the largest tile gets only 3%) and is **8.5× larger** (RMS 0.39).

**C. R3 reanalysis:** with random z, the error rises in **84–98% of tiles** for 3 of the 4 in-range targets (spread evenly: entropy 0.86–0.93). On the fourth (8117), known and random z are equally poor (that target wasn't recovered well even with known z). So wrong-z errors are global, as hypothesized.

**Interpretation and implications**
- In this drift generator, **the discrete noise labels carry most of the "what" of an image** (scene composition, viewpoint, palette, texture), and ε mainly adjusts local detail within the scene z selects.
- **Standard CSGM optimizes only the weak half of the latent.** With a randomly drawn z, CSGM must push ε far off-distribution to imitate a different scene (Check 3b: per-patch norms up to 4.8×), which is where tiles and wrong textures appear. This explains Check 5 R3 (wrong z → wrong content everywhere) and why the noise-label search (Check 7) helps, and suggests stronger z search or a continuous relaxation of the conditioning vector as the natural next step for inverting drift generators.
- Caveats: 8 classes, 8 codes and 8 ε per class; the descriptors are coarse; shares depend on the descriptors chosen.